### Polni vnos

- Obstoječe podatke zamenja z novim stanjem
- Ohrani le eno verzijo, ki jo prepiše z vsemi spremembami (posodobitve, brisanja, dodajanja)

In [4]:
import dlt
from datetime import datetime, timezone
from uuid import uuid4

from dlt.destinations import filesystem
from mongodb import mongodb

In [11]:
SOURCE_SYSTEM = "mongodb_atlas"

DATABASE_NAME = "test"

BUCKET_NAME = "book-library"

INGESTION_TYPE = "full"

INGESTED_AT = datetime.now(timezone.utc)

RUN_ID = str(uuid4())

In [12]:
source = mongodb(
    database=DATABASE_NAME
)

print("Collections:")

for resource_name in source.resources:
    print("-", resource_name)

Collections:
- authors
- books


In [13]:
def add_metadata(resource_name):

    def mapper(item):

        item = dict(item)

        item["_source_system"] = SOURCE_SYSTEM

        item["_source_resource"] = resource_name

        item["_ingestion_type"] = INGESTION_TYPE

        item["_ingested_at"] = INGESTED_AT

        item["_load_id"] = RUN_ID

        return item

    return mapper


for resource_name, resource in source.resources.items():

    resource.add_map(
        add_metadata(resource_name)
    )

print("Metadata READY")

Metadata READY


In [14]:
destination = filesystem(
    bucket_url=f"s3://{BUCKET_NAME}",

    layout=(
        "{table_name}/"
        "ingestion_date={YYYY}-{MM}-{DD}/"
        "{load_id}.{file_id}.{ext}"
    ),

    current_datetime=INGESTED_AT
)

In [15]:
pipeline = dlt.pipeline(
    pipeline_name="mongodb_to_silo",

    destination=destination,

    dataset_name=SOURCE_SYSTEM
)

print("Pipeline READY")

Pipeline READY


In [16]:
info = pipeline.run(
    source,

    write_disposition="replace",

    loader_file_format="parquet"
)

print(info)

Pipeline mongodb_to_silo load step finished in 0.20 seconds
1 load package(s) were loaded to destination filesystem and into dataset mongodb_atlas
The filesystem destination used s3://book-library location to store data
Load package 1793650519.1912434 is LOADED and contains no failed jobs


Spremembe v jezeru: zamenja se .parquet datoteka z novo, posodobljeno verzijo